# 注意力

In [1]:
import torch

## 乘法注意力

In [2]:
def dot_product_attention(query, keys, values):
    # query: (hidden_size,)
    # keys: (src_len, hidden_size)
    # values: (src_len, hidden_size)
    
    # 计算得分: 点积
    scores = torch.matmul(keys, query)  # (src_len,)
    
    # Softmax归一化
    attention_weights = torch.softmax(scores, dim=0)
    
    # 加权求和
    context = torch.matmul(attention_weights, values)
    
    return context, attention_weights

## 加法注意力

In [ ]:
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        # 可学习的参数矩阵
        self.W_query = nn.Linear(hidden_size, hidden_size, bias=False)
        self.W_keys = nn.Linear(hidden_size, hidden_size, bias=False)
        self.v = nn.Linear(hidden_size, 1, bias=False)
    
    def forward(self, query, keys, values):
        # query: (batch, hidden)
        # keys: (batch, src_len, hidden)
        # values: (batch, src_len, hidden)
        
        batch_size, src_len, _ = keys.shape
        
        # 扩展query维度以便广播
        query_expanded = self.W_query(query).unsqueeze(1)  # (batch, 1, hidden)
        
        # 计算能量分数
        keys_transformed = self.W_keys(keys)  # (batch, src_len, hidden)
        energy = torch.tanh(query_expanded + keys_transformed)  # (batch, src_len, hidden)
        
        # 计算最终得分
        scores = self.v(energy).squeeze(-1)  # (batch, src_len)
        
        # Softmax归一化
        attention_weights = torch.softmax(scores, dim=1)
        
        # 加权求和
        context = torch.bmm(attention_weights.unsqueeze(1), values).squeeze(1)
        
        return context, attention_weights

## 缩放乘法注意力

In [ ]:
def scaled_dot_product_attention(query, keys, values, mask=None):
    # query: (batch, hidden)
    # keys: (batch, src_len, hidden)
    
    d_k = query.size(-1)
    
    # 计算得分并缩放
    scores = torch.matmul(query.unsqueeze(1), keys.transpose(-2, -1)) / math.sqrt(d_k)
    scores = scores.squeeze(1)  # (batch, src_len)
    
    # 可选：应用mask（padding位置设为-inf）
    if mask is not None:
        scores = scores.masked_fill(mask == 0, -1e10)
    
    attention_weights = torch.softmax(scores, dim=-1)
    
    context = torch.bmm(attention_weights.unsqueeze(1), values).squeeze(1)
    
    return context, attention_weights

# Transformer